<a href="https://colab.research.google.com/github/RossIsland/MINLP-Surrogate-Modelling/blob/main/pipeline_2_ANN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Install PyTorch, Torchvision, and Pyomo (removed the non-existent scalar_formatter)
!pip install -q torch torchvision pyomo

# 2. Install amplpy AND ampltools
!pip install -q amplpy ampltools

# 3. Install the CBC solver module directly via amplpy
!python -m amplpy.modules install cbc

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 8.3 MB/s eta 0:00:00
$ /usr/bin/python3 -m pip install -i https://pypi.ampl.com ampl_module_base ampl_module_cbc
Looking in indexes: https://pypi.ampl.com
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 71.6 MB/s eta 0:00:00
Imported ampl_module_base.
Imported ampl_module_base.
Imported ampl_module_cbc.


In [ ]:
# ==========================================
# CELL 2: DATA GENERATION & PREPROCESSING
# ==========================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split

def simulate_pipeline_physics(P_in, T_in, Q, d, L, delta_h):
    g = 9.81
    R_M = 518.0
    lambda_f = 0.012

    elevation_term = (g * delta_h) / (R_M * T_in)
    friction_term = (lambda_f * L * (Q**2)) / (d**5 * 1e12)

    P_out = P_in - (friction_term * 0.5 + elevation_term * 0.1)
    T_out = T_in - (0.00002 * L) + (0.05 * (P_in - P_out))

    P_out = np.clip(P_out, 2.0, 12.0)
    T_out = np.clip(T_out, 260.0, 340.0)
    return P_out, T_out

np.random.seed(42)
num_samples = 8000

P_in_samples = np.random.uniform(4.0, 10.0, num_samples)
T_in_samples = np.random.uniform(273.15, 333.15, num_samples)
Q_samples = np.random.uniform(100.0, 1800.0, num_samples)
d_samples = np.random.uniform(1.0, 1.2, num_samples)
L_samples = np.random.uniform(10000, 100000, num_samples)
delta_h_samples = np.random.uniform(-50, 50, num_samples)

P_out_samples = []
T_out_samples = []

for i in range(num_samples):
    po, to = simulate_pipeline_physics(P_in_samples[i], T_in_samples[i], Q_samples[i],
                                      d_samples[i], L_samples[i], delta_h_samples[i])
    P_out_samples.append(po)
    T_out_samples.append(to)

df = pd.DataFrame({
    'P_in': P_in_samples, 'T_in': T_in_samples, 'Q': Q_samples,
    'd': d_samples, 'L': L_samples, 'delta_h': delta_h_samples,
    'P_out': P_out_samples, 'T_out': T_out_samples
})

X = df[['P_in', 'T_in', 'Q', 'd', 'L', 'delta_h']].values
Y = df[['P_out', 'T_out']].values

X_offset = X.mean(axis=0)
X_factor = X.std(axis=0)
Y_offset = Y.mean(axis=0)
Y_factor = Y.std(axis=0)

X_scaled = (X - X_offset) / X_factor
Y_scaled = (Y - Y_offset) / Y_factor

X_train, X_test, Y_train, Y_test = train_test_split(X_scaled, Y_scaled, test_size=0.2, random_state=42)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
Y_train_t = torch.tensor(Y_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
Y_test_t = torch.tensor(Y_test, dtype=torch.float32)

print("Data generation complete! Dataset shape:", df.shape)

Data generation complete! Dataset shape: (8000, 8)


In [ ]:
# ==========================================
# CELL 3: TRAINING THE ReLU NEURAL NETWORK
# ==========================================
class PipelineANN(nn.Module):
    def __init__(self):
        super(PipelineANN, self).__init__()
        self.fc1 = nn.Linear(6, 10)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(10, 2)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

model = PipelineANN()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

print("Training Pipeline Segment ANN...")
for epoch in range(100):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train_t)
    loss = criterion(outputs, Y_train_t)
    loss.backward()
    optimizer.step()

model.eval()
with torch.no_grad():
    test_preds = model(X_test_t)
    test_loss = criterion(test_preds, Y_test_t)
    print(f"Final Test MSE Loss: {test_loss.item():.5f}")

W1 = model.fc1.weight.detach().numpy()
b1 = model.fc1.bias.detach().numpy()
W2 = model.fc2.weight.detach().numpy()
b2 = model.fc2.bias.detach().numpy()

Training Pipeline Segment ANN...
Final Test MSE Loss: 0.00905


In [ ]:
# ==========================================
# CELL 4: PYOMO MATHEMATICAL OPTIMIZATION
# ==========================================
from pyomo.environ import *
from amplpy import modules

# 1. Initialize the Optimization Model
opt_model = ConcreteModel()

# 2. Define Variables with Initial Bounds
opt_model.P_in = Var(bounds=(4.0, 10.0), initialize=7.0)
opt_model.T_in = Var(bounds=(273.15, 333.15), initialize=300.0)
opt_model.Q = Var(bounds=(100.0, 1800.0), initialize=500.0)
opt_model.d = Var(bounds=(1.0, 1.2), initialize=1.1)
opt_model.L = Var(bounds=(10000, 100000), initialize=50000)
opt_model.delta_h = Var(bounds=(-50, 50), initialize=0.0)

opt_model.P_out = Var(bounds=(2.0, 12.0))
opt_model.T_out = Var(bounds=(260.0, 340.0))

opt_model.neurons = RangeSet(0, 9)
opt_model.u = Var(opt_model.neurons, bounds=(0.0, 50.0))
opt_model.delta = Var(opt_model.neurons, within=Binary)

M_ub = 100.0
M_lb = -100.0

# 3. Model Scaling & Structure Functions
def get_scaled_input(m):
    return [
        (m.P_in - X_offset[0]) / X_factor[0],
        (m.T_in - X_offset[1]) / X_factor[1],
        (m.Q - X_offset[2]) / X_factor[2],
        (m.d - X_offset[3]) / X_factor[3],
        (m.L - X_offset[4]) / X_factor[4],
        (m.delta_h - X_offset[5]) / X_factor[5]
    ]

def relu_bigm_rules(m, k):
    inputs = get_scaled_input(m)
    u_hat = sum(W1[k, i] * inputs[i] for i in range(6)) + b1[k]

    yield m.u[k] >= u_hat
    yield m.u[k] <= u_hat - M_lb * (1 - m.delta[k])
    yield m.u[k] <= M_ub * m.delta[k]
    yield u_hat >= M_lb * (1 - m.delta[k])

# 4. Generate Constraints
opt_model.relu_constraints = ConstraintList()
for k in opt_model.neurons:
    for c in relu_bigm_rules(opt_model, k):
        opt_model.relu_constraints.add(c)

def output_p_rule(m):
    y_scaled_p = sum(W2[0, k] * m.u[k] for k in m.neurons) + b2[0]
    return m.P_out == (y_scaled_p * Y_factor[0]) + Y_offset[0]
opt_model.out_p_con = Constraint(rule=output_p_rule)

def output_t_rule(m):
    y_scaled_t = sum(W2[1, k] * m.u[k] for k in m.neurons) + b2[1]
    return m.T_out == (y_scaled_t * Y_factor[1]) + Y_offset[1]
opt_model.out_t_con = Constraint(rule=output_t_rule)

# 5. Fix Independent Parameter Boundary States
opt_model.P_in.fix(8.5)
opt_model.T_in.fix(293.15)
opt_model.Q.fix(1200.0)
opt_model.d.fix(1.166)
opt_model.L.fix(50000)
opt_model.delta_h.fix(15.0)

opt_model.obj = Objective(expr=opt_model.P_out, sense=maximize)

# ========================================================
# ALTERNATIVE ROBUST SOLVER INTERFACE (AMPL METHODOLOGY)
# ========================================================

# Locate the actual executable path for the CBC binary
executable_path = modules.find("cbc")

# Define the factory using the universal ASL interface format ('cbcnl')
solver = SolverFactory("cbcnl", executable=executable_path, solve_io="nl")

# Execute and automatically load calculations directly back to model variables
results = solver.solve(opt_model)

# ========================================================
# PRINT RESULTS
# ========================================================
print("\n--- Pipeline Optimization Result via Embedded ReLU-ANN ---")
print(f"Fixed Input Pressure:  {value(opt_model.P_in):.2f} MPa")
print(f"Optimized Predicted Outlet Pressure: {value(opt_model.P_out):.4f} MPa")
print(f"Optimized Predicted Outlet Temp:     {value(opt_model.T_out):.2f} K")


--- Pipeline Optimization Result via Embedded ReLU-ANN ---
Fixed Input Pressure:  8.50 MPa
Optimized Predicted Outlet Pressure: 8.6137 MPa
Optimized Predicted Outlet Temp:     291.21 K


In [ ]:
# ==============================================================================
# CELL: EXTRACT LOCALIZED PHYSICAL LINEAR CONSTRAINTS FROM RELU-ANN (UNIT CORRECTED)
# ==============================================================================
import numpy as np

# --- STEP 1: CAPTURE ACTIVE NEURONS FROM THE SOLVED PYOMO MODEL ---
active_neurons = []
for k in opt_model.neurons:
    if value(opt_model.delta[k]) > 0.5:
        active_neurons.append(k)

print(f"Active ReLU hidden neurons at this operating point: {active_neurons}")

# --- STEP 2: COLLAPSE THE ANN WEIGHTS INTO STANDARD MATRIX RELATIONSHIPS ---
# Inputs: [P_in, T_in, Q, d, L, delta_h]
# Outputs: Index 0 -> P_out, Index 1 -> T_out

m_scaled_P = np.zeros(6)
c_scaled_P = b2[0]

m_scaled_T = np.zeros(6)
c_scaled_T = b2[1]

# Accumulate slope coefficients and intercepts from the active neurons
for k in active_neurons:
    # Contribution to P_out
    m_scaled_P += W2[0, k] * W1[k, :]
    c_scaled_P += W2[0, k] * b1[k]

    # Contribution to T_out
    m_scaled_T += W2[1, k] * W1[k, :]
    c_scaled_T += W2[1, k] * b1[k]

# --- STEP 3: RECONVERT FROM STANDARDIZED SPACE TO RAW PHYSICAL UNITS ---
# Calculate continuous physical space slopes and intercepts
m_phys_P = (m_scaled_P / X_factor) * Y_factor[0]
c_phys_P = Y_offset[0] + Y_factor[0] * (c_scaled_P - sum((m_scaled_P * X_offset) / X_factor))

m_phys_T = (m_scaled_T / X_factor) * Y_factor[1]
c_phys_T = Y_offset[1] + Y_factor[1] * (c_scaled_T - sum((m_scaled_T * X_offset) / X_factor))

# --- STEP 4: MATHEMATICAL UNIT MAPPING TO THE PAPER'S BASELINE GRID SPACE ---
# Volumetric flow conversion factor: 10^4 m3/d directly to standard operational m3/s
flow_conversion = 10000.0 / 86400.0

# Adjust the input gradient coefficient matching variable x[2] (which tracks Q)
m_phys_P[2] = m_phys_P[2] * flow_conversion
m_phys_T[2] = m_phys_T[2] * flow_conversion

# --- STEP 5: DISPLAY CONSTRAINTS IN READABLE ALGEBRAIC FORM ---
print("\n" + "="*70)
print("UNIT-CORRECTED EXTRACTED LINEAR CONSTRAINTS FOR GUROBI SOLVER")
print("="*70)
print("Variables mapping context:")
print("  x[0] = P_in, x[1] = T_in, x[2] = Q (10^4 m3/d), x[3] = d, x[4] = L, x[5] = delta_h\n")

print("1. Pressure Output Constraint Equation (P_out):")
print(f"   P_out = ({m_phys_P[0]:.6e} * P_in) + ")
print(f"           ({m_phys_P[1]:.6e} * T_in) + ")
print(f"           ({m_phys_P[2]:.6e} * Q) + ")
print(f"           ({m_phys_P[3]:.6e} * d) + ")
print(f"           ({m_phys_P[4]:.6e} * L) + ")
print(f"           ({m_phys_P[5]:.6e} * delta_h) + ({c_phys_P:.6f})")

print("\n2. Temperature Output Constraint Equation (T_out):")
print(f"   T_out = ({m_phys_T[0]:.6e} * P_in) + ")
print(f"           ({m_phys_T[1]:.6e} * T_in) + ")
print(f"           ({m_phys_T[2]:.6e} * Q) + ")
print(f"           ({m_phys_T[3]:.6e} * d) + ")
print(f"           ({m_phys_T[4]:.6e} * L) + ")
print(f"           ({m_phys_T[5]:.6e} * delta_h) + ({c_phys_T:.6f})")
print("="*70)

Active ReLU hidden neurons at this operating point: [0, 3, 4, 5, 6, 7]

UNIT-CORRECTED EXTRACTED LINEAR CONSTRAINTS FOR GUROBI SOLVER
Variables mapping context:
  x[0] = P_in, x[1] = T_in, x[2] = Q (10^4 m3/d), x[3] = d, x[4] = L, x[5] = delta_h

1. Pressure Output Constraint Equation (P_out):
   P_out = (1.187822e+00 * P_in) + 
           (2.825055e-02 * T_in) + 
           (-1.647939e-05 * Q) + 
           (1.859381e+00 * d) + 
           (-1.113134e-06 * L) + 
           (-2.395774e-04 * delta_h) + (-11.702359)

2. Temperature Output Constraint Equation (T_out):
   T_out = (2.385662e+00 * P_in) + 
           (9.103613e-01 * T_in) + 
           (3.415186e-05 * Q) + 
           (3.659100e+01 * d) + 
           (-3.246226e-05 * L) + 
           (2.051846e-02 * delta_h) + (-37.646961)


In [ ]:
# --- STEP 6: EXTRACT THE CORRESPONDING NUMERICAL VALUE OF DESIGN VARIABLE 'd' ---
try:
    # If 'd' is an indexed variable or parameter per pipeline segment/arc
    # Replace '1' with your active segment/arc index if needed
    d_value = value(opt_model.d[1])
    print(f"Extracted Value of d (Indexed): {d_value} m")
except Exception:
    try:
        # If 'd' is a single global scalar variable or parameter in your model
        d_value = value(opt_model.d)
        print(f"Extracted Value of d (Scalar): {d_value} m")
    except Exception as e:
        print("Could not automatically extract 'd'. Check your Pyomo model component names.")
        print(f"Error details: {e}")

Extracted Value of d (Scalar): 1.166 m


In [ ]:
# --- STEP 7: EXTRACT NUMERICAL VALUES OF DESIGN/OPERATING PARAMETERS L AND delta_h ---
print("\n" + "-"*40)
print("EXTRACTED OPERATING CONFIGURATION VALUES")
print("-"*40)

# 1. Extract Pipeline Segment Length (L)
try:
    # Tracks if L is indexed per segment/arc container
    L_value = value(opt_model.L[1]) # Replace '1' with your active segment/arc index if needed
    print(f"Pipeline Length (L)       : {L_value:.2f} m")
except Exception:
    try:
        # Tracks if L is declared as a scalar param/var
        L_value = value(opt_model.L)
        print(f"Pipeline Length (L)       : {L_value:.2f} m")
    except Exception:
        print("Pipeline Length (L)       : Variable/Parameter token not found in opt_model.")

# 2. Extract Elevation Head Profile Profile Change (delta_h)
try:
    # Tracks if delta_h is indexed per segment/arc container
    dh_value = value(opt_model.delta_h[1]) # Replace '1' with your active segment/arc index if needed
    print(f"Elevation Profile (delta_h): {dh_value:.2f} m")
except Exception:
    try:
        # Tracks if delta_h is declared as a scalar param/var
        dh_value = value(opt_model.delta_h)
        print(f"Elevation Profile (delta_h): {dh_value:.2f} m")
    except Exception:
        print("Elevation Profile (delta_h): Variable/Parameter token not found in opt_model.")
print("-"*40)


----------------------------------------
EXTRACTED OPERATING CONFIGURATION VALUES
----------------------------------------
Pipeline Length (L)       : 50000.00 m
Elevation Profile (delta_h): 15.00 m
----------------------------------------
